<h2>SQL QUERY</h2>

<h3>Load Datasets& Data Cleaning</h3>

In [6]:
import pandas as pd
import numpy as np
import sqlite3
import requests
import time

In [8]:
def load_socrata_dataset(base_url, params, limit=25000, sleep_time=0.2):
    all_rows = []
    offset = 0
    page = 1

    while True:
        page_params = params.copy()
        page_params["$limit"] = limit
        page_params["$offset"] = offset

        print(f"Requesting page {page}, offset {offset}...")

        response = requests.get(base_url, params=page_params, timeout=120)
        response.raise_for_status()

        rows = response.json()

        if len(rows) == 0:
            print("Download complete.")
            break

        all_rows.extend(rows)
        print(f"Downloaded {len(all_rows):,} rows so far.")

        offset += limit
        page += 1
        time.sleep(sleep_time)

    return pd.DataFrame(all_rows)

In [9]:
crime_url = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"

crime_params = {
    "$select": (
        "id, date, primary_type, description, location_description, "
        "arrest, domestic, beat, district, ward, community_area, latitude, longitude"
    ),
    "$where": "date >= '2020-01-01T00:00:00' AND date < '2025-01-01T00:00:00'",
    "$order": "date, id"
}

crime = load_socrata_dataset(crime_url, crime_params)

# Remove exact duplicate rows that may appear during API pagination
crime = crime.drop_duplicates().copy()

print("Crime dataset shape:", crime.shape)

Requesting page 1, offset 0...
Downloaded 25,000 rows so far.
Requesting page 2, offset 25000...
Downloaded 50,000 rows so far.
Requesting page 3, offset 50000...
Downloaded 75,000 rows so far.
Requesting page 4, offset 75000...
Downloaded 100,000 rows so far.
Requesting page 5, offset 100000...
Downloaded 125,000 rows so far.
Requesting page 6, offset 125000...
Downloaded 150,000 rows so far.
Requesting page 7, offset 150000...
Downloaded 175,000 rows so far.
Requesting page 8, offset 175000...
Downloaded 200,000 rows so far.
Requesting page 9, offset 200000...
Downloaded 225,000 rows so far.
Requesting page 10, offset 225000...
Downloaded 250,000 rows so far.
Requesting page 11, offset 250000...
Downloaded 275,000 rows so far.
Requesting page 12, offset 275000...
Downloaded 300,000 rows so far.
Requesting page 13, offset 300000...
Downloaded 325,000 rows so far.
Requesting page 14, offset 325000...
Downloaded 350,000 rows so far.
Requesting page 15, offset 350000...
Downloaded 375,00

In [10]:
socio_url = "https://data.cityofchicago.org/resource/kn9c-c2s2.json"

socio_params = {
    "$select": (
        "ca, community_area_name, percent_households_below_poverty, "
        "per_capita_income_, hardship_index"
    )
}

socio_raw = load_socrata_dataset(socio_url, socio_params, limit=5000)

socio = socio_raw.rename(columns={
    "ca": "community_area",
    "percent_households_below_poverty": "poverty_rate",
    "per_capita_income_": "per_capita_income"
})

print("Socioeconomic dataset shape:", socio.shape)

Requesting page 1, offset 0...
Downloaded 78 rows so far.
Requesting page 2, offset 5000...
Download complete.
Socioeconomic dataset shape: (78, 5)


In [12]:
crime_clean = crime.copy()

crime_columns = [
    "id", "date", "primary_type", "description", "location_description",
    "arrest", "domestic", "beat", "district", "ward",
    "community_area", "latitude", "longitude"
]

for col in crime_columns:
    if col not in crime_clean.columns:
        crime_clean[col] = np.nan

crime_clean = crime_clean[crime_columns]

# Convert date
crime_clean["date"] = pd.to_datetime(crime_clean["date"], errors="coerce")

# Convert numeric columns
numeric_crime_columns = [
    "id", "beat", "district", "ward",
    "community_area", "latitude", "longitude"
]

for col in numeric_crime_columns:
    crime_clean[col] = pd.to_numeric(crime_clean[col], errors="coerce")

# Convert arrest and domestic to 0/1
crime_clean["arrest"] = crime_clean["arrest"].astype(str).str.lower().map({
    "true": 1,
    "false": 0,
    "1": 1,
    "0": 0
})

crime_clean["domestic"] = crime_clean["domestic"].astype(str).str.lower().map({
    "true": 1,
    "false": 0,
    "1": 1,
    "0": 0
})

# Remove records without community_area
crime_clean = crime_clean.dropna(subset=["community_area"]).copy()
crime_clean["community_area"] = crime_clean["community_area"].astype(int)

# Create time-based features
crime_clean["year"] = crime_clean["date"].dt.year
crime_clean["month"] = crime_clean["date"].dt.month
crime_clean["weekday"] = crime_clean["date"].dt.day_name()
crime_clean["hour"] = crime_clean["date"].dt.hour

crime_clean["is_weekend"] = crime_clean["weekday"].isin(
    ["Saturday", "Sunday"]
).astype(int)

# Create violent crime indicator
violent_crimes = [
    "HOMICIDE",
    "CRIMINAL SEXUAL ASSAULT",
    "CRIM SEXUAL ASSAULT",
    "ROBBERY",
    "ASSAULT",
    "BATTERY",
    "KIDNAPPING"
]

crime_clean["is_violent"] = crime_clean["primary_type"].isin(
    violent_crimes
).astype(int)

print("crime_clean shape:", crime_clean.shape)

crime_clean shape: (1184892, 19)


In [13]:
socio_clean = socio.copy()

socio_columns = [
    "community_area",
    "community_area_name",
    "poverty_rate",
    "per_capita_income",
    "hardship_index"
]

for col in socio_columns:
    if col not in socio_clean.columns:
        socio_clean[col] = np.nan

socio_clean = socio_clean[socio_columns]

# Convert numeric columns
numeric_socio_columns = [
    "community_area",
    "poverty_rate",
    "per_capita_income",
    "hardship_index"
]

for col in numeric_socio_columns:
    socio_clean[col] = pd.to_numeric(socio_clean[col], errors="coerce")

# Keep only Chicago's 77 community areas
# The API includes one citywide summary row, which is not a community area
socio_clean = socio_clean[
    socio_clean["community_area"].between(1, 77)
].copy()

socio_clean["community_area"] = socio_clean["community_area"].astype(int)

print("socio_clean shape:", socio_clean.shape)

socio_clean shape: (77, 5)


In [14]:
conn = sqlite3.connect("chicago_crime_project.db")

crime_clean.to_sql("crime", conn, if_exists="replace", index=False)
socio_clean.to_sql("socio", conn, if_exists="replace", index=False)

def run_sql(query):
    return pd.read_sql_query(query, conn)

print("SQLite tables created: crime, socio")

SQLite tables created: crime, socio


In [15]:
conn.execute("DROP TABLE IF EXISTS crime_area_summary;")

conn.execute("""
CREATE TABLE crime_area_summary AS
SELECT
    community_area,
    COUNT(*) AS total_crimes,
    SUM(is_violent) AS violent_crimes,
    1.0 * SUM(is_violent) / COUNT(*) AS violent_crime_share,
    AVG(arrest) AS arrest_rate,
    AVG(domestic) AS domestic_share
FROM crime
GROUP BY community_area;
""")

conn.commit()

print("SQLite summary table created: crime_area_summary")

SQLite summary table created: crime_area_summary


<h3>Query 1: Crime Table Size Validation</h3>

In [18]:
run_sql('''
SELECT
    COUNT(*) AS total_crime_records
FROM crime;
''')

,total_crime_records
0,1184892


The crime database contains 1,184,892 total records, confirming successful loading of all cleaned incident-level data. This count serves as the baseline for assessing data completeness throughout the analysis.

<h3>Query 2: Community Area Coverage</h3>

In [35]:
run_sql("""
SELECT
    (SELECT COUNT(DISTINCT community_area) FROM crime) AS communities_in_crime_data,
    (SELECT COUNT(DISTINCT community_area) FROM socio) AS communities_in_socio_data,
    (SELECT COUNT(DISTINCT community_area) FROM crime_area_summary) AS communities_in_summary;
""")

,communities_in_crime_data,communities_in_socio_data,communities_in_summary
0,77,77,77


All 77 Chicago community areas are represented in the crime data, socioeconomic data, and aggregated summary. Complete community coverage ensures the subsequent analysis covers the entire city without geographic gaps, enabling valid neighborhood-level comparisons.

<h3>Query 3: Aggregation Summary Statistics</h3>

In [21]:
run_sql('''
SELECT
    COUNT(*) AS number_of_community_areas,
    ROUND(AVG(total_crimes), 1) AS avg_total_crimes,
    MIN(total_crimes) AS min_total_crimes,
    MAX(total_crimes) AS max_total_crimes,
    ROUND(AVG(violent_crime_share), 4) AS avg_violent_crime_share,
    ROUND(AVG(arrest_rate), 4) AS avg_arrest_rate
FROM crime_area_summary;
''')

,number_of_community_areas,avg_total_crimes,min_total_crimes,max_total_crimes,avg_violent_crime_share,avg_arrest_rate
0,77,15388.2,1396,62510,0.3057,0.1185


The community-level aggregation reveals substantial variation across the 77 neighborhoods. Average total crimes per community are 15,388, but this masks wide range: from 1,396 crimes (Edison Park) to 62,510 crimes (Austin). Average violent crime share of 0.3057 (30.6%) indicates violent crimes constitute roughly one-third of all reported incidents. Variation in arrest rates (average 0.1185 or 11.85%) may reflect differences in police enforcement, crime types, or reporting patterns across neighborhoods.

<h2> Query 4: Top 5 Communities by Total Crime with Socioeconomic Context

In [22]:
run_sql('''
SELECT
    s.community_area_name,
    c.total_crimes,
    c.violent_crimes,
    ROUND(c.violent_crime_share, 4) AS violent_crime_share,
    ROUND(s.hardship_index, 1) AS hardship_index,
    ROUND(s.poverty_rate, 1) AS poverty_rate
FROM crime_area_summary c
JOIN socio s ON c.community_area = s.community_area
ORDER BY c.total_crimes DESC
LIMIT 5;
''')

,community_area_name,total_crimes,violent_crimes,violent_crime_share,hardship_index,poverty_rate
0,Austin,62510,23642,0.3782,73.0,28.6
1,Near North Side,49009,11957,0.2440,1.0,12.9
2,Near West Side,44421,12018,0.2705,15.0,20.6
3,South Shore,40968,15509,0.3786,55.0,31.1
4,Loop,36932,8583,0.2324,3.0,14.7


The five highest-crime communities—Austin (62,510 crimes), Near North Side (49,009), Near West Side (44,421), South Shore (40,968), and the Loop (36,932)—account for 233,840 crimes, or approximately 19.7% of Chicago's total reported incidents from 2020-2024. However, these communities show divergent socioeconomic profiles. Austin and South Shore have relatively high hardship indices of 73.0 and 55.0, while Near North Side and the Loop have very low hardship indices of 1.0 and 3.0. This heterogeneity suggests that crime concentration reflects multiple factors beyond economic disadvantage, including population density, commercial activity, visitor exposure, and land use patterns.

<h2> Query 5: Specific High-Crime, High-Hardship Communities

In [24]:
run_sql('''
SELECT
    s.community_area_name,
    c.total_crimes,
    ROUND(c.violent_crime_share, 4) AS violent_crime_share,
    ROUND(s.hardship_index, 1) AS hardship_index,
    ROUND(s.poverty_rate, 1) AS poverty_rate,
    ROUND(s.per_capita_income, 0) AS per_capita_income
FROM crime_area_summary c
JOIN socio s ON c.community_area = s.community_area
WHERE c.total_crimes > (SELECT AVG(total_crimes) FROM crime_area_summary)
  AND s.hardship_index > (SELECT AVG(hardship_index) FROM socio)
ORDER BY c.total_crimes DESC;
''')

,community_area_name,total_crimes,violent_crime_share,hardship_index,poverty_rate,per_capita_income
0,Austin,62510,0.3782,73.0,28.6,15957.0
1,South Shore,40968,0.3786,55.0,31.1,19398.0
2,North Lawndale,34464,0.3867,87.0,43.1,12034.0
3,Humboldt park,32980,0.3380,85.0,33.9,13781.0
4,Auburn Gresham,32221,0.3603,74.0,27.6,15528.0
5,Greater Grand Crossing,30880,0.3787,66.0,29.6,17285.0
6,Chatham,29184,0.3599,60.0,27.8,18881.0
7,Roseland,29146,0.3479,52.0,19.8,17949.0
8,Englewood,25044,0.4044,94.0,46.6,11888.0
9,Chicago Lawn,25004,0.3655,80.0,27.9,13231.0


The 20 high-crime, high-hardship communities include Austin, South Shore, North Lawndale, Humboldt Park, Auburn Gresham, and Greater Grand Crossing among the highest-crime neighborhoods in this group. These communities show both elevated crime and socioeconomic strain: poverty rates range from 18.7% to 46.6%, per capita income averages approximately $15,253, and hardship indices range from 52.0 to 96.0. Understanding the specific neighborhoods in this category is essential for targeted policy interventions that combine public safety investments with economic development strategies.

<h3> Query 6: Crime Profile by Hardship Group

In [26]:
run_sql('''
SELECT
    CASE
        WHEN s.hardship_index < 50 THEN 'Low Hardship'
        WHEN s.hardship_index >= 50 AND s.hardship_index < 74 THEN 'Mid Hardship'
        WHEN s.hardship_index >= 74 THEN 'High Hardship'
    END AS hardship_group,
    COUNT(*) AS num_communities,
    ROUND(AVG(c.total_crimes), 0) AS avg_total_crimes,
    ROUND(AVG(c.violent_crimes), 0) AS avg_violent_crimes,
    ROUND(AVG(c.violent_crime_share), 4) AS avg_violent_crime_share,
    ROUND(AVG(c.arrest_rate), 4) AS avg_arrest_rate,
    ROUND(AVG(c.domestic_share), 4) AS avg_domestic_share
FROM crime_area_summary c
JOIN socio s ON c.community_area = s.community_area
GROUP BY hardship_group
ORDER BY CASE
        WHEN hardship_group = 'Low Hardship' THEN 1
        WHEN hardship_group = 'Mid Hardship' THEN 2
        WHEN hardship_group = 'High Hardship' THEN 3
    END;
''')

,hardship_group,num_communities,avg_total_crimes,avg_violent_crimes,avg_violent_crime_share,avg_arrest_rate,avg_domestic_share
0,Low Hardship,38,13625.0,3538.0,0.2614,0.1007,0.1521
1,Mid Hardship,19,17259.0,6041.0,0.3273,0.1199,0.2321
2,High Hardship,20,16961.0,6321.0,0.3692,0.1508,0.2441


Crime patterns exhibit a clear relationship with hardship levels. Low-hardship communities average 13,625 total crimes and 3,538 violent crimes, with a 26.1% violent crime share. Mid-hardship communities average 17,259 total crimes and 6,041 violent crimes, with a 32.7% violent crime share. High-hardship communities average 16,961 total crimes and 6,321 violent crimes, with the highest violent crime share at 36.9%. Arrest rates also increase with hardship, from 10.1% in low-hardship communities to 12.0% in mid-hardship communities and 15.1% in high-hardship communities. Domestic-related crime share also rises from 15.2% in low-hardship areas to 24.4% in high-hardship areas, suggesting that economically stressed communities tend to have a larger share of domestic-related incidents.

<h3> Query 7: High-volume but lower-violent-share communities

In [37]:
run_sql("""
SELECT
    s.community_area_name,
    c.total_crimes,
    ROUND(c.violent_crime_share, 4) AS violent_crime_share,
    ROUND(s.hardship_index, 1) AS hardship_index,
    ROUND(s.poverty_rate, 1) AS poverty_rate,
    ROUND(s.per_capita_income, 0) AS per_capita_income
FROM crime_area_summary c
JOIN socio s ON c.community_area = s.community_area
WHERE c.total_crimes > (SELECT AVG(total_crimes) FROM crime_area_summary)
  AND c.violent_crime_share < (SELECT AVG(violent_crime_share) FROM crime_area_summary)
ORDER BY c.total_crimes DESC
LIMIT 10;
""")

,community_area_name,total_crimes,violent_crime_share,hardship_index,poverty_rate,per_capita_income
0,Near North Side,49009,0.2440,1.0,12.9,88669.0
1,Near West Side,44421,0.2705,15.0,20.6,44689.0
2,Loop,36932,0.2324,3.0,14.7,65526.0
3,West Town,34249,0.2330,10.0,14.7,43198.0
4,Lake View,28361,0.2203,5.0,11.4,60058.0
5,Logan Square,22217,0.2437,23.0,16.8,31908.0
6,Uptown,19193,0.3055,20.0,24.0,35787.0
7,West Ridge,17912,0.2824,46.0,17.2,23040.0
8,Lincoln Park,17696,0.1686,2.0,12.3,71551.0


This query identifies communities with above-average total reported crime but below-average violent crime share. These areas, including Near North Side, Near West Side, the Loop, West Town, and Lake View, have high crime volume but relatively lower violent-crime composition. This supports the finding that total reported crime volume alone does not fully describe neighborhood safety, because some high-volume areas may reflect commercial activity, visitor exposure, or reporting density rather than a more severe violent-crime profile.

<h3> Query 8: Low-volume but higher-violent-share communities

In [39]:
run_sql("""
SELECT
    s.community_area_name,
    c.total_crimes,
    c.violent_crimes,
    ROUND(c.violent_crime_share, 4) AS violent_crime_share,
    ROUND(s.hardship_index, 1) AS hardship_index,
    ROUND(s.poverty_rate, 1) AS poverty_rate,
    ROUND(s.per_capita_income, 0) AS per_capita_income
FROM crime_area_summary c
JOIN socio s ON c.community_area = s.community_area
WHERE c.total_crimes < (SELECT AVG(total_crimes) FROM crime_area_summary)
  AND c.violent_crime_share > (SELECT AVG(violent_crime_share) FROM crime_area_summary)
ORDER BY c.violent_crime_share DESC
LIMIT 10;
""")

,community_area_name,total_crimes,violent_crimes,violent_crime_share,hardship_index,poverty_rate,per_capita_income
0,Riverdale,5948,2510,0.4220,98.0,56.5,8201.0
1,Fuller Park,3322,1281,0.3856,97.0,51.2,10432.0
2,Washington Park,11239,4270,0.3799,88.0,42.1,13785.0
3,Brighton Park,10278,3848,0.3744,84.0,23.6,13089.0
4,Gage Park,10291,3667,0.3563,93.0,23.4,12171.0
5,Armour Square,5185,1794,0.3460,82.0,40.1,16148.0
6,East Side,5919,1995,0.3371,64.0,19.2,17104.0
7,South Deering,8852,2925,0.3304,65.0,29.2,14685.0
8,Hermosa,6885,2273,0.3301,71.0,20.5,15089.0
9,Lower West Side,13397,4422,0.3301,76.0,25.8,16444.0


This query identifies communities with below-average total reported crime but above-average violent crime share. These communities may not appear at the top of total-crime rankings, but violent incidents account for a relatively large share of their reported crimes. This complements Query 7 by showing the opposite pattern: lower crime volume can still be associated with a more severe crime composition.

<h3> Query 9: Communities Ranked by Violent Crime Share

In [30]:
run_sql('''
SELECT
    ROW_NUMBER() OVER (ORDER BY c.violent_crime_share DESC) AS violent_crime_rank,
    s.community_area_name,
    c.total_crimes,
    c.violent_crimes,
    ROUND(c.violent_crime_share, 4) AS violent_crime_share,
    ROUND(s.hardship_index, 1) AS hardship_index
FROM crime_area_summary c
JOIN socio s ON c.community_area = s.community_area
ORDER BY c.violent_crime_share DESC;
''')

,violent_crime_rank,community_area_name,total_crimes,violent_crimes,violent_crime_share,hardship_index
0,1,Riverdale,5948,2510,0.4220,98.0
1,2,Englewood,25044,10127,0.4044,94.0
2,3,New City,19511,7687,0.3940,91.0
3,4,South Lawndale,19502,7669,0.3932,96.0
4,5,North Lawndale,34464,13326,0.3867,87.0
...,...,...,...,...,...,...
72,73,Edison Park,1396,286,0.2049,8.0
73,74,O'Hare,8063,1605,0.1991,24.0
74,75,North Center,6452,1228,0.1903,6.0
75,76,Forest Glen,2739,487,0.1778,11.0


This ranking prioritizes communities by violent crime proportion, independent of total crime volume. Communities with high violent crime share face disproportionate severe-crime challenges. The ranking shows that violent crime share varies from approximately 16.9% in Lincoln Park to 42.2% in Riverdale. Communities with both high total crime and high violent crime share face compounded public safety challenges. This perspective complements total crime rankings because it identifies neighborhoods where violence represents a particularly large share of crime, even when absolute crime volume is lower.

<h3> Query 10: Extreme Community Comparison - Top 10 vs Bottom 10

In [33]:
# Top 10 highest-crime communities
run_sql('''
SELECT
    s.community_area_name,
    c.total_crimes,
    ROUND(c.violent_crime_share, 4) AS violent_crime_share,
    ROUND(s.hardship_index, 1) AS hardship_index,
    ROUND(s.poverty_rate, 1) AS poverty_rate,
    ROUND(s.per_capita_income, 0) AS per_capita_income
FROM crime_area_summary c
JOIN socio s ON c.community_area = s.community_area
ORDER BY c.total_crimes DESC
LIMIT 10;
''')

,community_area_name,total_crimes,violent_crime_share,hardship_index,poverty_rate,per_capita_income
0,Austin,62510,0.3782,73.0,28.6,15957.0
1,Near North Side,49009,0.2440,1.0,12.9,88669.0
2,Near West Side,44421,0.2705,15.0,20.6,44689.0
3,South Shore,40968,0.3786,55.0,31.1,19398.0
4,Loop,36932,0.2324,3.0,14.7,65526.0
5,North Lawndale,34464,0.3867,87.0,43.1,12034.0
6,West Town,34249,0.2330,10.0,14.7,43198.0
7,Humboldt park,32980,0.3380,85.0,33.9,13781.0
8,Auburn Gresham,32221,0.3603,74.0,27.6,15528.0
9,Greater Grand Crossing,30880,0.3787,66.0,29.6,17285.0


In [34]:
# Bottom 10 lowest-crime communities
run_sql('''
SELECT
    s.community_area_name,
    c.total_crimes,
    ROUND(c.violent_crime_share, 4) AS violent_crime_share,
    ROUND(s.hardship_index, 1) AS hardship_index,
    ROUND(s.poverty_rate, 1) AS poverty_rate,
    ROUND(s.per_capita_income, 0) AS per_capita_income
FROM crime_area_summary c
JOIN socio s ON c.community_area = s.community_area
ORDER BY c.total_crimes ASC
LIMIT 10;
''')

,community_area_name,total_crimes,violent_crime_share,hardship_index,poverty_rate,per_capita_income
0,Edison Park,1396,0.2049,8.0,3.3,40959.0
1,Burnside,1668,0.3237,79.0,33.0,12515.0
2,Mount Greenwood,2665,0.2458,16.0,3.4,34381.0
3,Forest Glen,2739,0.1778,11.0,7.5,44164.0
4,Montclaire,3137,0.2608,50.0,15.3,22014.0
5,Hegewisch,3321,0.3014,44.0,17.1,22677.0
6,Fuller Park,3322,0.3856,97.0,51.2,10432.0
7,Oakland,3840,0.3187,78.0,39.7,19252.0
8,McKinley Park,4337,0.3129,61.0,18.7,16954.0
9,West Elsdon,4432,0.2884,69.0,15.6,15754.0


The extreme community comparison starkly illustrates neighborhood inequality. The top 10 highest-crime communities average 39,863 crimes per community, with a 32.0% violent crime share, an average hardship index of 46.9, and average per capita income of $33,606. However, this group is socioeconomically heterogeneous: Austin and South Shore have higher hardship indices of 73.0 and 55.0, while Near North Side and the Loop are economically advantaged, with hardship indices of 1.0 and 3.0. The bottom 10 lowest-crime communities average only 3,086 crimes per community, with a 28.2% violent crime share. Their average hardship index is 51.3, slightly above the city average, and average per capita income is $23,910, which is substantially lower than the high-crime group despite lower crime volume. This surprising pattern suggests that factors beyond poverty alone determine crime patterns. The 12.9x difference in crime between the top and bottom groups demonstrates the magnitude of neighborhood variation in public safety challenges.

## SQL Summary


The SQL analysis supports the main findings from the Python notebook by validating the cleaned database, aggregating incident-level crime records to Chicago’s 77 community areas, and joining crime profiles with socioeconomic indicators. The queries show that reported crime is highly concentrated across communities, but high total crime volume does not always represent the same type of neighborhood risk.

The SQL results also show why crime composition matters. Some high-volume communities, such as Near North Side, the Loop, and West Town, have relatively lower violent-crime shares, suggesting that total crime volume may partly reflect commercial activity, visitor exposure, or reporting density. In contrast, communities such as Riverdale and Fuller Park have lower total crime counts but higher violent-crime shares, meaning they could be overlooked if neighborhoods are ranked only by total incidents. The hardship-group query further shows that higher-hardship communities tend to have higher violent-crime and domestic-related crime shares.

Overall, the SQL queries reinforce the main conclusion that neighborhood safety should be evaluated using both crime volume and crime composition, together with socioeconomic context.